# 🎙️ Tetrax ASR: Fine-Tuning Whisper Medium on IndicVoices for Marathi + Hindi

Welcome to the official **Tetrax ASR Bilingual Fine-Tuning Notebook**.

This notebook trains and adapts **`openai/whisper-medium`** (769M multilingual parameters) using **PEFT / LoRA (Low-Rank Adaptation)** on the **AI4Bharat IndicVoices** dataset to build a high-accuracy speech recognition model for **Marathi (`mr`)** and **Hindi (`hi`)**.

### 🚀 Memory Optimization for Free Google Colab T4 GPU (15GB VRAM):
- Fine-tunes **`openai/whisper-medium`** using parameter-efficient LoRA (`r=32`, `lora_alpha=64`).
- Trains only ~1.1% of parameters (~8.3M params) while freezing the core 769M acoustic backbone.
- `per_device_train_batch_size = 1` with `gradient_accumulation_steps = 8` (effective batch size = 8).
- `fp16 = True` and `gradient_checkpointing = True` to fit comfortably in 15GB VRAM without Out-Of-Memory (OOM) errors.
- Evaluates both **WER** (Word Error Rate) and **CER** (Character Error Rate for Devanagari script).

---

## Step 1: Check GPU & Install Dependencies

In [ ]:
# Verify GPU allocation (T4 GPU with ~15GB VRAM recommended)
!nvidia-smi

# Install required fine-tuning and evaluation libraries
!pip install -q transformers datasets peft accelerate evaluate jiwer soundfile librosa bitsandbytes tensorboard

# Colab pre-installs an outdated torchao (0.10.0) which conflicts with PEFT (requires >=0.16.0)
# Uninstalling torchao allows PEFT to use standard LoRA linear adapters smoothly
!pip uninstall -y torchao


## Step 2: Mount Google Drive
Mount your drive to access your manually uploaded IndicVoices dataset in `/content/drive/MyDrive/Tetrax-ASR/data/` and permanently save your fine-tuned Whisper Medium checkpoints.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DATA_DIR = "/content/drive/MyDrive/Tetrax-ASR/data"
CHECKPOINT_DIR = "/content/drive/MyDrive/Tetrax-ASR/checkpoints/tetrax-whisper-medium-marathi-hindi"
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Google Drive mounted successfully.")
print(f"📁 Data directory: {DRIVE_DATA_DIR}")
print(f"💾 Checkpoints directory: {CHECKPOINT_DIR}")

## Step 3: Load AI4Bharat IndicVoices Dataset (Marathi + Hindi)
This cell automatically scans your Google Drive (`/content/drive/MyDrive/Tetrax-ASR/data/`) or `/content/` for manually placed parquet files:
1. Checks subfolders: `data/marathi/` and `data/hindi/`
2. Checks filename patterns: `*marathi*.parquet` and `*hindi*.parquet` (or generic `*.parquet`)
3. If local files are not found, it downloads directly from Hugging Face Hub (`ai4bharat/IndicVoices`).
4. Creates a balanced, stratified 80% Train / 10% Validation / 10% Test split across Marathi and Hindi.

In [ ]:
import glob
import os
from datasets import load_dataset, Audio, DatasetDict, concatenate_datasets

DRIVE_DATA = '/content/drive/MyDrive/Tetrax-ASR/data'
LOCAL_DATA = '/content'

# --------------------------------------------------------------------------------
# DATASET SLICING FOR COLAB T4 GPU:
# You downloaded 73 Marathi files (35GB) and 83 Hindi files (46GB) to Google Drive!
# Colab Free has 12.7GB System RAM. Loading 2-4 parquet files per language (~1-2 GB)
# gives ~10-25 hours of audio, perfect for high-accuracy LoRA fine-tuning.
# Set PARQUET_FILES_PER_LANG = None if you want to train on ALL files.
# --------------------------------------------------------------------------------
PARQUET_FILES_PER_LANG = 2   # Set to 1-4 for fast training, or None for all
MAX_SAMPLES_PER_LANG = None  # Optional: cap total samples per language

def find_lang_parquets(lang_name, lang_code):
    patterns = [
        f'{DRIVE_DATA}/{lang_name}/*.parquet',
        f'{DRIVE_DATA}/{lang_name}/**/*.parquet',
        f'{DRIVE_DATA}/{lang_code}/*.parquet',
        f'{DRIVE_DATA}/*{lang_name}*.parquet',
        f'{DRIVE_DATA}/*{lang_code}*.parquet',
        f'{LOCAL_DATA}/{lang_name}/*.parquet',
        f'{LOCAL_DATA}/*{lang_name}*.parquet',
    ]
    files = []
    for p in patterns:
        files.extend(glob.glob(p, recursive=True))
    return sorted(list(dict.fromkeys(files)))

mr_files = find_lang_parquets('marathi', 'mr')
hi_files = find_lang_parquets('hindi', 'hi')

print(f'Total Marathi parquet files found in Drive: {len(mr_files)}')
print(f'Total Hindi parquet files found in Drive: {len(hi_files)}')

if PARQUET_FILES_PER_LANG:
    mr_files_to_load = mr_files[:PARQUET_FILES_PER_LANG] if mr_files else []
    hi_files_to_load = hi_files[:PARQUET_FILES_PER_LANG] if hi_files else []
    print(f'Loading subset of {len(mr_files_to_load)} Marathi and {len(hi_files_to_load)} Hindi file(s) for memory safety.')
else:
    mr_files_to_load = mr_files
    hi_files_to_load = hi_files
    print('Loading ALL available parquet files...')

# 1. Load Marathi dataset
if mr_files_to_load:
    print(f'Loading Marathi parquet files: {mr_files_to_load}')
    ds_mr = load_dataset('parquet', data_files=mr_files_to_load, split='train')
else:
    print('Loading Marathi from Hugging Face Hub (ai4bharat/IndicVoices, marathi)...')
    ds_mr = load_dataset('ai4bharat/IndicVoices', 'marathi', split='train')

if 'language' not in ds_mr.column_names:
    ds_mr = ds_mr.add_column('language', ['mr'] * len(ds_mr))

# 2. Load Hindi dataset
if hi_files_to_load:
    print(f'Loading Hindi parquet files: {hi_files_to_load}')
    ds_hi = load_dataset('parquet', data_files=hi_files_to_load, split='train')
else:
    print('Loading Hindi from Hugging Face Hub (ai4bharat/IndicVoices, hindi)...')
    ds_hi = load_dataset('ai4bharat/IndicVoices', 'hindi', split='train')

if 'language' not in ds_hi.column_names:
    ds_hi = ds_hi.add_column('language', ['hi'] * len(ds_hi))

# Optional sample capping
if MAX_SAMPLES_PER_LANG:
    if len(ds_mr) > MAX_SAMPLES_PER_LANG:
        ds_mr = ds_mr.select(range(MAX_SAMPLES_PER_LANG))
    if len(ds_hi) > MAX_SAMPLES_PER_LANG:
        ds_hi = ds_hi.select(range(MAX_SAMPLES_PER_LANG))

# Normalize audio columns to decode=False for speed & stability
def normalize_audio_col(ds):
    for col in ['audio', 'audio_filepath']:
        if col in ds.column_names:
            try:
                return ds.cast_column(col, Audio(decode=False))
            except Exception:
                pass
    return ds

ds_mr = normalize_audio_col(ds_mr)
ds_hi = normalize_audio_col(ds_hi)

# Combine into bilingual dataset
raw_dataset = concatenate_datasets([ds_mr, ds_hi])
print(f'\nTotal active samples: {len(raw_dataset)} ({len(ds_mr)} Marathi, {len(ds_hi)} Hindi)')

# Stratified 80/10/10 split across languages
raw_dataset = raw_dataset.class_encode_column('language')
first_split = raw_dataset.train_test_split(test_size=0.20, seed=42, stratify_by_column='language')
train_ds = first_split['train']

second_split = first_split['test'].train_test_split(test_size=0.50, seed=42, stratify_by_column='language')
val_ds = second_split['train']
test_ds = second_split['test']

dataset_dict = DatasetDict({
    'train': train_ds,
    'validation': val_ds,
    'test': test_ds
})

print(f'\n✅ Balanced bilingual splits created:')
print(f'  • Train samples: {len(train_ds)}')
print(f'  • Validation samples: {len(val_ds)}')
print(f'  • Test samples: {len(test_ds)}')


### Inspect Sample Marathi and Hindi Audio & Devanagari Transcripts

In [ ]:
import io
import os
import soundfile as sf
import IPython.display as ipd

def get_sample_audio_array(sample):
    # Prioritize 'audio' dict (bytes or array)
    for key in ['audio', 'audio_filepath']:
        val = sample.get(key)
        if isinstance(val, dict):
            if val.get('bytes'):
                return sf.read(io.BytesIO(val['bytes']))
            if val.get('array') is not None:
                return val['array'], val.get('sampling_rate', 16000)
            if val.get('path') and os.path.exists(str(val['path'])):
                return sf.read(val['path'])
        elif isinstance(val, (str, os.PathLike)) and os.path.exists(str(val)):
            return sf.read(str(val))
    return None, None

for idx in [0, len(train_ds) // 2]:
    sample = train_ds[idx]
    lang_val = "mr" if sample.get("language") == 0 else "hi"
    text_val = sample.get("text", sample.get("normalized_text", ""))
    print(f"\n--- Sample #{idx} [{lang_val.upper()}] ---")
    print("Transcript:", text_val)
    print("Duration:", sample.get("duration"), "sec")

    audio_data, sr = get_sample_audio_array(sample)
    if audio_data is not None:
        ipd.display(ipd.Audio(audio_data, rate=sr))


## Step 4: Setup Multilingual Whisper Medium Processor
We initialize the WhisperProcessor for **`openai/whisper-medium`** configured for multilingual transcription.

In [ ]:
from transformers import WhisperProcessor

MODEL_ID = "openai/whisper-medium"
print(f"Initializing WhisperProcessor for {MODEL_ID}...")

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="mr", task="transcribe")
print("✅ Whisper Medium processor initialized.")

## Step 5: Bilingual Preprocessing & Dynamic Language Tokenization
Computes 80-channel log-Mel spectrograms from 16kHz audio, and dynamically conditions the tokenizer tokens for each sample (`<|mr|>` for Marathi, `<|hi|>` for Hindi).

In [ ]:
import io
import os
import soundfile as sf
import librosa
import numpy as np

def extract_raw_audio(batch):
    """
    Robustly extracts audio array and sample rate from any Hugging Face Audio dict,
    parquet binary bytes, or disk file path.
    """
    # 1. Check audio candidate columns (prioritize 'audio' over 'audio_filepath' string)
    for col in ['audio', 'audio_filepath', 'audio_path', 'file', 'path']:
        if col in batch and batch[col] is not None:
            entry = batch[col]
            # Dict with pre-decoded array
            if isinstance(entry, dict) and entry.get('array') is not None:
                return np.asarray(entry['array'], dtype=np.float32), entry.get('sampling_rate', 16000)
            # Dict with raw bytes
            if isinstance(entry, dict) and entry.get('bytes'):
                arr, sr = sf.read(io.BytesIO(entry['bytes']), dtype='float32')
                return arr, sr
            # Dict with file path on disk
            if isinstance(entry, dict) and entry.get('path') and os.path.exists(str(entry['path'])):
                arr, sr = sf.read(entry['path'], dtype='float32')
                return arr, sr
            # Direct bytes
            if isinstance(entry, bytes) and len(entry) > 0:
                arr, sr = sf.read(io.BytesIO(entry), dtype='float32')
                return arr, sr
            # Direct string path on disk
            if isinstance(entry, (str, os.PathLike)) and os.path.exists(str(entry)):
                arr, sr = sf.read(str(entry), dtype='float32')
                return arr, sr

    # 2. Check relative paths in data search roots
    for col in ['audio_filepath', 'audio', 'path']:
        if col in batch and batch[col] is not None:
            val = batch[col]
            path_str = val.get('path') if isinstance(val, dict) else (val if isinstance(val, str) else None)
            if path_str:
                for base in [DRIVE_DATA, LOCAL_DATA, '/content', '/content/drive/MyDrive/Tetrax-ASR/data']:
                    candidate = os.path.join(base, path_str)
                    if os.path.exists(candidate):
                        arr, sr = sf.read(candidate, dtype='float32')
                        return arr, sr

    batch_summary = {k: (type(v).__name__, list(v.keys()) if isinstance(v, dict) else str(v)[:60]) for k, v in batch.items()}
    raise ValueError(f'Could not decode audio column format. Available fields: {batch_summary}')

def prepare_bilingual_dataset(batch):
    # 1. Decode audio to 16kHz float32 mono
    audio_arr, orig_sr = extract_raw_audio(batch)

    if audio_arr.ndim > 1:
        audio_arr = audio_arr.mean(axis=1)
    if orig_sr != 16000:
        audio_arr = librosa.resample(audio_arr, orig_sr=orig_sr, target_sr=16000)

    # 2. Extract log-Mel spectrogram (80 channels)
    batch['input_features'] = processor.feature_extractor(
        audio_arr,
        sampling_rate=16000
    ).input_features[0]

    # 3. Dynamic language prompt prefixing (mr vs hi)
    sample_lang = 'mr' if batch.get('language') in (0, 'mr', 'marathi') else 'hi'
    processor.tokenizer.set_prefix_tokens(language=sample_lang, task='transcribe')
    text_content = batch.get('text') or batch.get('normalized_text') or ''
    batch['labels'] = processor.tokenizer(text_content).input_ids
    return batch

print('Mapping features and tokenizing labels across splits...')
tokenized_train = dataset_dict['train'].map(
    prepare_bilingual_dataset,
    remove_columns=dataset_dict['train'].column_names,
    num_proc=2
)
tokenized_val = dataset_dict['validation'].map(
    prepare_bilingual_dataset,
    remove_columns=dataset_dict['validation'].column_names,
    num_proc=2
)
print('✅ Bilingual preprocessing finished successfully!')


## Step 6: Data Collator and Devanagari Evaluation Metrics (WER + CER)
We compute both **Word Error Rate (WER)** and **Character Error Rate (CER)** to accurately measure recognition performance on Devanagari text.

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}

## Step 7: Load Whisper Medium (769M) & Apply PEFT LoRA with Gradient Checkpointing

In [ ]:
import sys
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model
import peft.import_utils

print(f"Loading WhisperForConditionalGeneration ({MODEL_ID}) in FP16...")
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# Enable gradient checkpointing to fit Whisper-Medium within Colab's 15GB T4 GPU
model.gradient_checkpointing_enable()

# LoRA Configuration targeting attention projections
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none"
)

# Bypass Colab's outdated pre-installed torchao (<0.16.0) across all loaded PEFT modules
peft.import_utils.is_torchao_available = lambda: False
for mod in list(sys.modules.values()):
    if hasattr(mod, "is_torchao_available"):
        try:
            setattr(mod, "is_torchao_available", lambda: False)
        except Exception:
            pass

model = get_peft_model(model, peft_config)
print("\n📊 Trainable Parameters Summary:")
model.print_trainable_parameters()


## Step 8: Fine-Tune with Seq2SeqTrainer (T4 GPU Memory Optimized)
- `per_device_train_batch_size = 1`
- `gradient_accumulation_steps = 8` (effective batch size = 8)
- `gradient_checkpointing = True`
- `fp16 = True`

In [ ]:
import os
import gc
import torch
import inspect
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# 1. Verify prerequisites from previous steps
if 'tokenized_train' not in globals() or 'tokenized_val' not in globals():
    raise NameError("❌ `tokenized_train` and `tokenized_val` are missing! The Colab runtime was restarted. Please run Step 5 (Preprocessing) first.")
if 'model' not in globals():
    raise NameError("❌ `model` is missing! Please run Step 7 (Load Whisper Medium & LoRA) first.")
if 'data_collator' not in globals():
    raise NameError("❌ `data_collator` is missing! Please run Step 6 (Data Collator & Metrics) first.")

# 2. Free any fragmented CUDA cache from previous runs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "TETRAX-ASR-WHISPER-MEDIUM-769M"
OUTPUT_DIR = f"/content/{MODEL_NAME}"

# 3. Use a representative subset of validation data (200 samples) for fast intermediate eval
# Slices evaluation time from 15 minutes down to ~30 seconds and avoids RAM/VRAM exhaustion
eval_subset = tokenized_val.select(range(min(200, len(tokenized_val))))

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,       # Effective batch size = 8
    learning_rate=1e-4,
    warmup_steps=100,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=200,                       # Evaluate every 200 steps
    save_steps=200,
    logging_steps=25,
    fp16=True,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,           # Offloads eval batches to CPU RAM immediately (prevents CUDA OOM)
    generation_max_length=128,           # Constrains max generated tokens to save memory
    gradient_checkpointing=True,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
)

def preprocess_logits_for_metrics(logits, labels):
    """
    Reduces 51,865-dimensional vocabulary logits to 1D token IDs directly on GPU,
    reducing evaluation memory footprint by >50,000x and preventing 5.7GB allocation spikes.
    """
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

trainer_kwargs = {
    "args": training_args,
    "model": model,
    "train_dataset": tokenized_train,
    "eval_dataset": eval_subset,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "preprocess_logits_for_metrics": preprocess_logits_for_metrics,
}

if "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters:
    trainer_kwargs["processing_class"] = processor.feature_extractor
else:
    trainer_kwargs["tokenizer"] = processor.feature_extractor

trainer = Seq2SeqTrainer(**trainer_kwargs)

# Check if previous checkpoint exists to resume automatically
resume_ckpt = None
if os.path.exists(OUTPUT_DIR):
    ckpts = [os.path.join(OUTPUT_DIR, d) for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume_ckpt = max(ckpts, key=os.path.getmtime)
        print(f"🔄 Found existing checkpoint: {resume_ckpt}. Resuming training...")

print("🚀 Starting Whisper Medium bilingual fine-tuning on Marathi + Hindi...")
trainer.train(resume_from_checkpoint=resume_ckpt)


## Step 9: Save Fine-Tuned LoRA Adapter & Export to Google Drive & ZIP

In [ ]:
import shutil
import os
from google.colab import files

MODEL_NAME = "TETRAX-ASR-WHISPER-MEDIUM-769M"
OUTPUT_DIR = f"/content/{MODEL_NAME}"
ZIP_NAME = f"{MODEL_NAME}.zip"

# 1. Save LoRA adapter weights & processor
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"✅ Saved fine-tuned LoRA adapter to: {OUTPUT_DIR}")

# 2. Package into a compressed ZIP file
shutil.make_archive(MODEL_NAME, "zip", OUTPUT_DIR)
zip_size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024) if os.path.exists(ZIP_NAME) else 0.0
print(f"📦 Created compressed ZIP: {ZIP_NAME} ({zip_size_mb:.2f} MB)")

# 3. Save directly to Google Drive
DRIVE_DIR = "/content/drive/MyDrive/Tetrax-ASR/checkpoints"
drive_zip_path = os.path.join(DRIVE_DIR, ZIP_NAME)
drive_folder_path = os.path.join(DRIVE_DIR, MODEL_NAME)

try:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copy2(ZIP_NAME, drive_zip_path)
    shutil.copytree(OUTPUT_DIR, drive_folder_path, dirs_exist_ok=True)
    # Also create compatibility copy
    compat_zip_path = os.path.join(DRIVE_DIR, "tetrax-whisper-medium-marathi-hindi.zip")
    shutil.copy2(ZIP_NAME, compat_zip_path)
    compat_folder_path = os.path.join(DRIVE_DIR, "tetrax-whisper-medium-marathi-hindi")
    shutil.copytree(OUTPUT_DIR, compat_folder_path, dirs_exist_ok=True)
except Exception as e:
    print(f"⚠️ Notice copying to Drive ({e}). Make sure drive.mount('/content/drive') was run.")

# 4. Print Training Success Banner
print("\n" + "=" * 72)
print("🎉 MODEL SUCCESSFULLY TRAINED AND SAVED!")
print(f"  • Model Name    : {MODEL_NAME}")
print("  • Parameter Size: 769M Parameters (~0.8B Backbone, ~8.3M Trainable LoRA)")
print("  • Languages     : Marathi ('mr') + Hindi ('hi') [Devanagari text]")
print(f"  • Saved to Drive: {drive_zip_path}")
print(f"  • ZIP File Size : {zip_size_mb:.2f} MB")
print("=" * 72 + "\n")

# 5. Trigger direct browser download
print("⬇️ Triggering direct browser download of ZIP archive...")
try:
    files.download(ZIP_NAME)
except Exception as e:
    print("Browser download notice:", e)


## 🎉 Deployment into your Tetrax ASR Project

### Step 1: Download or copy the model from Google Drive
Your model is saved on Google Drive at:
`MyDrive/Tetrax-ASR/checkpoints/TETRAX-ASR-WHISPER-MEDIUM-769M.zip`

### Step 2: Extract into your local repo
Extract the ZIP into:
`Tetrax-ASR/checkpoints/TETRAX-ASR-WHISPER-MEDIUM-769M/`

### Step 3: Start your Tetrax ASR service
```bash
python -m uvicorn tetrax_asr.api.main:app --host 0.0.0.0 --port 8000
```

### Step 4: Transcribe in Python or REST API
```python
from tetrax_asr.inference.engine import TetraxWhisperEngine
engine = TetraxWhisperEngine()
result = engine.transcribe('sample.wav', language='mr')  # Or language='hi'
print('Transcription:', result['text'])
```
